# Orchestrator Agent Deep Dive

Explore how the orchestrator uses LangGraph as a supervisor pattern with A2A client tools to coordinate research sub-agents.
The orchestrator implements the harness inner loop, using A2A to coordinate sub-agents.

## 1. Load Environment

In [ ]:
import os
import json
import httpx
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)
print("✅ Environment loaded")

## 2. Orchestrator Graph Structure

The orchestrator uses a linear pipeline with conditional review loop:

```
plan → ingest → research → write → review → finalize
                                       │
                                       └── (if score < 7.0) → write (retry)
```

In [ ]:
import sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "agents", "orchestrator"))

from agents.orchestrator.tools import a2a_discover_agents, plan_research_strategy

# Test research planning
plan = plan_research_strategy(
    query="Analyze the architecture of modern document processing systems",
    has_document=True,
)
print(f"Research plan: {plan}")
print("✅ Orchestrator planning works")

## 3. A2A Agent Discovery

The orchestrator discovers available sub-agents via their AgentCard endpoints.

In [ ]:
sub_agent_urls = [
    "http://localhost:8101",
    "http://localhost:8102",
    "http://localhost:8103",
    "http://localhost:8104",
]

discovered = a2a_discover_agents(sub_agent_urls)
print(f"Discovered {len(discovered)} agents:")
for agent in discovered:
    print(f"  - {agent['name']}: {agent['description'][:60]}")
    print(f"    Skills: {agent['skills']}")

if not discovered:
    print("⚠️ No agents discovered. Ensure they are running.")
else:
    print("\n✅ Agent discovery successful")

## 4. End-to-End Orchestration Test

Send a research request through the full orchestrator pipeline.

In [ ]:
from agents.orchestrator.tools import a2a_send_message

# Step-by-step execution matching the orchestrator graph
query = "What are the main contributions and methodology described in the paper?"

print("Step 1: Research (via Researcher agent)")
research_result = a2a_send_message("http://localhost:8102", query)
print(f"  Result length: {len(research_result)} chars")
print(f"  Preview: {research_result[:200]}...")

print("\nStep 2: Write (via Writer agent)")
write_input = f"Query: {query}\n\nResearch Context:\n{research_result[:2000]}"
write_result = a2a_send_message("http://localhost:8103", write_input)
print(f"  Result length: {len(write_result)} chars")
print(f"  Preview: {write_result[:200]}...")

print("\nStep 3: Review (via Reviewer agent)")
review_result = a2a_send_message("http://localhost:8104", write_result[:3000])
print(f"  Review: {review_result[:300]}")

print("\n✅ Orchestration pipeline executed successfully")